# Embed MedQA Release v3 with BGE-M3

Upload the Kaggle dataset folder that contains `kaggle_embedding_input.jsonl`. This notebook writes `embeddings.npy`, `chunk_ids.json`, and `embedding_manifest.json` to `/kaggle/working`.

In [ ]:
%pip install -q sentence-transformers

In [ ]:
from pathlib import Path
import gc
import json
import os
import time

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"
INPUT_NAME = "kaggle_embedding_input.jsonl"
BATCH_SIZE = 64
MAX_ROWS = 0  # set small value for a smoke test, 0 = full file
NORMALIZE_EMBEDDINGS = True
OUTPUT_DIR = Path("/kaggle/working")

def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)

def find_input() -> Path:
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        if INPUT_NAME in files:
            candidates.append(Path(root) / INPUT_NAME)
    if not candidates:
        for root, _, files in os.walk("/kaggle/input"):
            if "chunk_texts_for_embed.jsonl" in files:
                candidates.append(Path(root) / "chunk_texts_for_embed.jsonl")
    if not candidates:
        raise FileNotFoundError("Upload a Kaggle dataset containing kaggle_embedding_input.jsonl")
    return sorted(candidates)[0]

INPUT_PATH = find_input()
log(f"Input: {INPUT_PATH}")

In [ ]:
ids = []
texts = []
seen = set()

with open(INPUT_PATH, "r", encoding="utf-8") as fh:
    for line_no, raw in enumerate(fh, start=1):
        if not raw.strip():
            continue
        row = json.loads(raw)
        chunk_id = str(row["id"])
        if chunk_id in seen:
            raise ValueError(f"Duplicate chunk id at line {line_no}: {chunk_id}")
        seen.add(chunk_id)
        ids.append(chunk_id)
        texts.append(str(row["text"]))
        if MAX_ROWS and len(ids) >= MAX_ROWS:
            break

if not ids:
    raise ValueError("No rows loaded")

avg_chars = sum(len(t) for t in texts) / len(texts)
log(f"Loaded {len(texts)} chunks | avg_chars={avg_chars:.0f}")
log(f"Sample id: {ids[0]}")
log(f"Sample text: {texts[0][:180].replace(chr(10), ' ')}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Device: {device}")
if device == "cuda":
    log(f"GPU: {torch.cuda.get_device_name(0)}")

model = SentenceTransformer(MODEL_NAME, device=device)
dim = int(model.get_sentence_embedding_dimension())
log(f"Model loaded: {MODEL_NAME} | dim={dim}")

_ = model.encode(["warmup"], normalize_embeddings=NORMALIZE_EMBEDDINGS, show_progress_bar=False)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

embeddings = np.empty((len(texts), dim), dtype=np.float32)
total = len(texts)
total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
t0 = time.time()

for batch_index, start in enumerate(range(0, total, BATCH_SIZE), start=1):
    batch = texts[start:start + BATCH_SIZE]
    vecs = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    embeddings[start:start + len(batch)] = vecs.astype(np.float32, copy=False)
    if batch_index == 1 or batch_index == total_batches or batch_index % 50 == 0:
        done = min(start + len(batch), total)
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed else 0.0
        eta = (total - done) / rate / 60 if rate else 0.0
        log(f"Progress {done}/{total} ({done/total*100:.1f}%) | batch {batch_index}/{total_batches} | rate={rate:.0f} vec/s | ETA={eta:.1f}m")

elapsed = time.time() - t0
log(f"Embedding complete | shape={embeddings.shape} | time={elapsed/60:.1f}m")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
emb_path = OUTPUT_DIR / "embeddings.npy"
ids_path = OUTPUT_DIR / "chunk_ids.json"
manifest_path = OUTPUT_DIR / "embedding_manifest.json"

np.save(emb_path, embeddings)
with open(ids_path, "w", encoding="utf-8") as fh:
    json.dump(ids, fh, ensure_ascii=False)

manifest = {
    "model_name": MODEL_NAME,
    "input_path": str(INPUT_PATH),
    "chunk_count": len(ids),
    "vector_shape": list(embeddings.shape),
    "dtype": str(embeddings.dtype),
    "batch_size": BATCH_SIZE,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "elapsed_seconds": round(elapsed, 2),
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

log(f"Saved: {emb_path}")
log(f"Saved: {ids_path}")
log(f"Saved: {manifest_path}")
log("Download these three files from Kaggle Output and place them under rag-data/embeddings/staging/medqa_release_v3_all_open_enriched/multilingual/")